# **Final ROBUST04 Submission - Enhanced with RRF Fusion**

## Approach:
1. **Multi-RM3 RRF Fusion** - Combine top 5 RM3 configurations
2. **BM25 Ensemble** - Multiple BM25 parameter sets
3. **Query-Specific Tuning** - Adapt parameters by query characteristics
4. **Validation** - Test on 50 training queries first
5. **Final Submission** - Apply best to all 249 queries

## Current Best (from optimization):
- Single best config: 0.0546 MAP (+9.86%)
- Target with enhancements: 0.056-0.058 MAP (+12-17%)

## Cell 1: Setup

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
from tqdm import tqdm
from pyserini.search.lucene import LuceneSearcher
from trectools import TrecQrel, TrecRun, TrecEval
from collections import defaultdict
from tabulate import tabulate
import os

drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/TEXT/FinalProject'
QUERIES_PATH = os.path.join(BASE_PATH, 'queriesROBUST.txt')
QRELS_PATH = os.path.join(BASE_PATH, 'qrels_50_Queries')

# Load ALL 249 queries
queries_df = pd.read_csv(QUERIES_PATH, sep='\t', header=None, names=['qid', 'text'], dtype={'qid': str})
ALL_QIDS = sorted(queries_df['qid'].tolist(), key=int)  # All 249
TRAIN_QIDS = ALL_QIDS[:50]  # First 50 for validation
QUERIES_DICT = dict(zip(queries_df.qid, queries_df.text))

searcher = LuceneSearcher.from_prebuilt_index('robust04')
qrels = TrecQrel(QRELS_PATH)  # Only for training validation

print(f"✅ Loaded {len(ALL_QIDS)} queries total")
print(f"   - Training: {len(TRAIN_QIDS)} queries (with qrels)")
print(f"   - Test: {len(ALL_QIDS) - len(TRAIN_QIDS)} queries (for submission)")

## Cell 2: Top RM3 Configurations (from grid search)

In [ ]:
# Top 5 RM3 configs from your optimization results
TOP_RM3_CONFIGS = [
    {'name': 'best', 'k1': 0.9, 'b': 0.4, 'fb_docs': 5, 'fb_terms': 20, 'alpha': 0.5, 'map': 0.0546},
    {'name': 'second', 'k1': 0.6, 'b': 0.4, 'fb_docs': 5, 'fb_terms': 20, 'alpha': 0.5, 'map': 0.0543},
    {'name': 'third', 'k1': 0.4, 'b': 0.5, 'fb_docs': 10, 'fb_terms': 30, 'alpha': 0.5, 'map': 0.0540},
    {'name': 'fourth', 'k1': 0.9, 'b': 0.4, 'fb_docs': 10, 'fb_terms': 20, 'alpha': 0.5, 'map': 0.0536},
    {'name': 'fifth', 'k1': 0.6, 'b': 0.4, 'fb_docs': 5, 'fb_terms': 20, 'alpha': 0.7, 'map': 0.0535},
]

print("Top 5 RM3 Configurations:")
for i, cfg in enumerate(TOP_RM3_CONFIGS, 1):
    print(f"{i}. k1={cfg['k1']}, b={cfg['b']}, fd={cfg['fb_docs']}, ft={cfg['fb_terms']}, α={cfg['alpha']} → MAP={cfg['map']:.4f}")

## Cell 3: Retrieval Functions

In [ ]:
def get_rm3_scores_with_config(query, config, k=1000):
    """Get scores with specific RM3 configuration"""
    searcher.set_bm25(k1=config['k1'], b=config['b'])
    searcher.set_rm3(
        fb_docs=config['fb_docs'],
        fb_terms=config['fb_terms'],
        original_query_weight=config['alpha']
    )
    hits = searcher.search(query, k=k)
    searcher.unset_rm3()
    return {h.docid: float(h.score) for h in hits}

def reciprocal_rank_fusion(score_dicts_list, k=60):
    """RRF: Combine multiple ranked lists"""
    rrf_scores = defaultdict(float)
    
    for score_dict in score_dicts_list:
        ranked = sorted(score_dict.items(), key=lambda x: x[1], reverse=True)
        for rank, (docid, _) in enumerate(ranked, start=1):
            rrf_scores[docid] += 1.0 / (k + rank)
    
    return dict(rrf_scores)

def classify_query(query_text):
    """Classify query by characteristics for adaptive parameters"""
    words = query_text.split()
    num_words = len(words)
    
    if num_words <= 3:
        return 'short'
    elif num_words >= 7:
        return 'long'
    else:
        return 'medium'

print("✅ Retrieval functions ready")

## Cell 4: Enhancement 1 - Multi-RM3 RRF Fusion

In [ ]:
def multi_rm3_rrf_fusion(query):
    """Combine top 5 RM3 configs with RRF"""
    all_scores = []
    
    for config in TOP_RM3_CONFIGS:
        scores = get_rm3_scores_with_config(query, config, k=1000)
        all_scores.append(scores)
    
    return reciprocal_rank_fusion(all_scores, k=60)

def evaluate_pipeline(pipeline_func, pipeline_name, queries_dict, qrels):
    """Evaluate a retrieval pipeline"""
    all_rows = []
    
    for qid, query_text in tqdm(queries_dict.items(), desc=pipeline_name):
        try:
            scores = pipeline_func(query_text)
            ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:1000]
            
            for rank, (docid, score) in enumerate(ranked, start=1):
                all_rows.append([qid, "Q0", docid, rank, score, pipeline_name])
        except Exception as e:
            print(f"Error on {qid}: {e}")
            continue
    
    run_file = f"{pipeline_name}_run.txt"
    pd.DataFrame(all_rows).to_csv(run_file, sep=' ', header=False, index=False)
    return TrecEval(TrecRun(run_file), qrels)

# Test on training set
print("\nTesting Multi-RM3 RRF Fusion on 50 training queries...\n")

te_fusion = evaluate_pipeline(
    multi_rm3_rrf_fusion,
    "multi_rm3_rrf",
    {qid: QUERIES_DICT[qid] for qid in TRAIN_QIDS},
    qrels
)

fusion_map = te_fusion.get_map()
print(f"\n📊 Multi-RM3 RRF Fusion Results:")
print(f"   MAP: {fusion_map:.4f}")
print(f"   vs Best single: {((fusion_map - 0.0546) / 0.0546 * 100):+.2f}%")
print(f"   vs Baseline: {((fusion_map - 0.0497) / 0.0497 * 100):+.2f}%")

## Cell 5: Enhancement 2 - Query-Specific Parameter Tuning

In [ ]:
# Query-specific configurations
QUERY_SPECIFIC_CONFIGS = {
    'short': {'k1': 0.9, 'b': 0.4, 'fb_docs': 10, 'fb_terms': 30, 'alpha': 0.3},  # More expansion for short
    'medium': {'k1': 0.9, 'b': 0.4, 'fb_docs': 5, 'fb_terms': 20, 'alpha': 0.5},  # Best overall
    'long': {'k1': 0.9, 'b': 0.4, 'fb_docs': 5, 'fb_terms': 10, 'alpha': 0.7},    # Less expansion for long
}

def query_adaptive_retrieval(query):
    """Adapt parameters based on query characteristics"""
    query_type = classify_query(query)
    config = QUERY_SPECIFIC_CONFIGS[query_type]
    return get_rm3_scores_with_config(query, config, k=1000)

# Test on training set
print("\nTesting Query-Adaptive Parameters on 50 training queries...\n")

# Show query distribution
query_types = [classify_query(QUERIES_DICT[qid]) for qid in TRAIN_QIDS]
print("Query type distribution:")
print(f"  Short: {query_types.count('short')}")
print(f"  Medium: {query_types.count('medium')}")
print(f"  Long: {query_types.count('long')}")

te_adaptive = evaluate_pipeline(
    query_adaptive_retrieval,
    "query_adaptive",
    {qid: QUERIES_DICT[qid] for qid in TRAIN_QIDS},
    qrels
)

adaptive_map = te_adaptive.get_map()
print(f"\n📊 Query-Adaptive Results:")
print(f"   MAP: {adaptive_map:.4f}")
print(f"   vs Best single: {((adaptive_map - 0.0546) / 0.0546 * 100):+.2f}%")
print(f"   vs Baseline: {((adaptive_map - 0.0497) / 0.0497 * 100):+.2f}%")

## Cell 6: Enhancement 3 - Combined RRF + Query-Adaptive

In [ ]:
def combined_approach(query):
    """Combine RRF fusion with query-adaptive retrieval"""
    # Get RRF fusion scores
    rrf_scores = multi_rm3_rrf_fusion(query)
    
    # Get query-adaptive scores
    adaptive_scores = query_adaptive_retrieval(query)
    
    # Fuse them with RRF
    return reciprocal_rank_fusion([rrf_scores, adaptive_scores], k=60)

# Test on training set
print("\nTesting Combined Approach (RRF + Adaptive) on 50 training queries...\n")

te_combined = evaluate_pipeline(
    combined_approach,
    "combined_rrf_adaptive",
    {qid: QUERIES_DICT[qid] for qid in TRAIN_QIDS},
    qrels
)

combined_map = te_combined.get_map()
print(f"\n📊 Combined Approach Results:")
print(f"   MAP: {combined_map:.4f}")
print(f"   vs Best single: {((combined_map - 0.0546) / 0.0546 * 100):+.2f}%")
print(f"   vs Baseline: {((combined_map - 0.0497) / 0.0497 * 100):+.2f}%")

## Cell 7: Select Best Approach

In [ ]:
# Compare all approaches
results = [
    ['Best Single RM3', 0.0546, '0.00%', get_rm3_scores_with_config],
    ['Multi-RM3 RRF Fusion', fusion_map, f"{((fusion_map - 0.0546) / 0.0546 * 100):+.2f}%", multi_rm3_rrf_fusion],
    ['Query-Adaptive', adaptive_map, f"{((adaptive_map - 0.0546) / 0.0546 * 100):+.2f}%", query_adaptive_retrieval],
    ['Combined (RRF+Adaptive)', combined_map, f"{((combined_map - 0.0546) / 0.0546 * 100):+.2f}%", combined_approach],
]

print("\n" + "="*80)
print("ENHANCEMENT COMPARISON (on 50 training queries)")
print("="*80)
print(tabulate(
    [[r[0], f"{r[1]:.4f}", r[2]] for r in results],
    headers=['Approach', 'MAP', 'vs Best Single'],
    tablefmt='fancy_grid'
))

# Select best
best_idx = max(range(len(results)), key=lambda i: results[i][1])
best_name, best_map, best_improvement, best_func = results[best_idx]

print(f"\n🏆 BEST APPROACH: {best_name}")
print(f"   MAP: {best_map:.4f}")
print(f"   Improvement over single best: {best_improvement}")
print(f"   Improvement over baseline: {((best_map - 0.0497) / 0.0497 * 100):+.2f}%")
print("="*80)

## Cell 8: Generate Final Submission for ALL 249 Queries

In [ ]:
print("\n" + "="*80)
print(f"GENERATING FINAL SUBMISSION WITH: {best_name}")
print("="*80)
print(f"\nApplying to all {len(ALL_QIDS)} queries...\n")

all_rows = []
checkpoint_interval = 50

for i, qid in enumerate(tqdm(ALL_QIDS, desc="Processing queries"), 1):
    try:
        query = QUERIES_DICT[qid]
        
        # Apply best approach
        if best_name == 'Best Single RM3':
            scores = get_rm3_scores_with_config(query, TOP_RM3_CONFIGS[0], k=1000)
        else:
            scores = best_func(query)
        
        # Rank documents
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:1000]
        
        for rank, (docid, score) in enumerate(ranked, start=1):
            all_rows.append([qid, "Q0", docid, rank, score, "final_submission"])
        
        # Checkpoint every 50 queries
        if i % checkpoint_interval == 0:
            checkpoint_file = f"checkpoint_{i}_queries.txt"
            pd.DataFrame(all_rows).to_csv(checkpoint_file, sep=' ', header=False, index=False)
            print(f"  ✓ Checkpoint: {i}/{len(ALL_QIDS)} queries saved")
    
    except Exception as e:
        print(f"\n❌ Error on query {qid}: {e}")
        continue

print(f"\n✅ Processed all {len(ALL_QIDS)} queries!")
print(f"   Total rows: {len(all_rows):,}")

## Cell 9: Validate and Save Final Submission

In [ ]:
print("\n" + "="*80)
print("VALIDATION CHECKS")
print("="*80)

df_submission = pd.DataFrame(all_rows, columns=['qid', 'Q0', 'docid', 'rank', 'score', 'run_name'])

# Check 1: Number of queries
unique_qids = df_submission['qid'].nunique()
print(f"\n1. Number of queries: {unique_qids}")
if unique_qids == 249:
    print("   ✅ PASS: All 249 queries present")
else:
    print(f"   ⚠️ WARNING: Expected 249, got {unique_qids}")

# Check 2: Documents per query
docs_per_query = df_submission.groupby('qid').size()
print(f"\n2. Documents per query:")
print(f"   Min: {docs_per_query.min()}")
print(f"   Max: {docs_per_query.max()}")
print(f"   Mean: {docs_per_query.mean():.1f}")
if docs_per_query.max() <= 1000 and docs_per_query.min() > 0:
    print("   ✅ PASS: All queries have ≤1000 documents")
else:
    print("   ⚠️ WARNING: Some queries have >1000 or 0 documents")

# Check 3: Ranking order (sample check)
sample_qid = ALL_QIDS[0]
sample_ranks = df_submission[df_submission['qid'] == sample_qid]['rank'].tolist()
if sample_ranks == list(range(1, len(sample_ranks) + 1)):
    print(f"\n3. Ranking order: ✅ PASS (checked {sample_qid})")
else:
    print(f"\n3. Ranking order: ⚠️ WARNING (checked {sample_qid})")

# Check 4: No duplicates per query
duplicates = df_submission.groupby(['qid', 'docid']).size().max()
print(f"\n4. Duplicate documents: {duplicates}")
if duplicates == 1:
    print("   ✅ PASS: No duplicates found")
else:
    print("   ⚠️ WARNING: Duplicate documents found")

# Check 5: Format validation
print(f"\n5. Format validation:")
print(f"   Columns: {list(df_submission.columns)}")
print(f"   Sample row: {df_submission.iloc[0].tolist()}")
print("   ✅ PASS: Format looks correct")

print("\n" + "="*80)
print("SAVING FINAL SUBMISSION")
print("="*80)

# Save final submission
output_file = os.path.join(BASE_PATH, 'final_submission_enhanced.txt')
df_submission.to_csv(output_file, sep=' ', header=False, index=False)

print(f"\n✅ FINAL SUBMISSION SAVED!")
print(f"   File: {output_file}")
print(f"   Size: {len(all_rows):,} lines ({len(all_rows) / 1000:.1f}KB approx)")
print(f"   Approach: {best_name}")
print(f"   Expected MAP: ~{best_map:.4f} (based on training validation)")
print(f"   Improvement: ~{((best_map - 0.0497) / 0.0497 * 100):+.1f}% vs baseline")
print("\n🎯 READY FOR SUBMISSION!")
print("="*80)

## Cell 10: Summary Statistics

In [ ]:
print("\n" + "="*80)
print("FINAL SUBMISSION SUMMARY")
print("="*80)

summary = [
    ['Total Queries', len(ALL_QIDS)],
    ['Training Queries (with qrels)', len(TRAIN_QIDS)],
    ['Test Queries (blind)', len(ALL_QIDS) - len(TRAIN_QIDS)],
    ['Documents per Query (target)', 1000],
    ['Total Rows', f"{len(all_rows):,}"],
    ['', ''],
    ['Baseline BM25 MAP', '0.0497'],
    ['Best Single RM3 MAP', '0.0546 (+9.86%)'],
    ['**Final Approach MAP**', f'**{best_map:.4f} ({((best_map - 0.0497) / 0.0497 * 100):+.2f}%)**'],
    ['Final Approach', best_name],
]

print(tabulate(summary, tablefmt='fancy_grid'))

print("\n📊 Enhancement Results on Training Set:")
enhancement_summary = [
    ['Single Best RM3', '0.0546', 'Baseline for enhancements'],
    ['Multi-RM3 RRF', f"{fusion_map:.4f}", f"{((fusion_map - 0.0546) / 0.0546 * 100):+.2f}%"],
    ['Query-Adaptive', f"{adaptive_map:.4f}", f"{((adaptive_map - 0.0546) / 0.0546 * 100):+.2f}%"],
    ['Combined', f"{combined_map:.4f}", f"{((combined_map - 0.0546) / 0.0546 * 100):+.2f}%"],
]
print(tabulate(enhancement_summary, headers=['Approach', 'MAP', 'vs Single'], tablefmt='fancy_grid'))

print("\n" + "="*80)
print("🎉 SUBMISSION GENERATION COMPLETE!")
print("="*80)